# Burn severity workflow
version 8.2

this notebook takes a file containing one or more burn extent polygons and perfoms the burn severity workflow on them. the out put is either single files per polygon or if desiered one merged geojson for all input polygons

note: combining the results of more that roughly 50 fires into one geojson results in a very large file that is more or less unuseable with a desktop computer and this is not adviseable. 

In [ ]:
# Make sure we can import the code
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import datacube
import geopandas as gpd
from datacube.utils.cog import write_cog
from datetime import date, timedelta

from dea_burn_severity.burn_severity_config import RuntimeBurnConfig


# Set database credetials here
config = RuntimeBurnConfig(
    db_name="fire_severity_product",
    db_host="db-aurora-dea-fire-severity.cluster-cxhoeczwhtar.ap-southeast-2.rds.amazonaws.com",
    # db_user="",
    # db_password="",
)


## Define input file

this is the file containing the burn extent polygons you wish to perform burn severity mapping on. Ic can be located in the 'gdata1' share drive or on your own scratch space. by default it is pointing at the 'early cut' of polygons provided by Aurora. 

Any (vector) file format will work, Esri Shapefile, geojson, json, geopackage

(if you are using an esri shapefile they come in four seperate file parts; .cpg, .dbf, .prj, .shp. all parts need to sit in the same folder but you point the script at the '.shp' part )


In [2]:
from dea_burn_severity.database import InputDatabase

database = InputDatabase(config)

all_boundaries_in_database = database.load_polygons_from_database()

Querying polygons from table 'Identifier('public', 'nli_lastboundaries_trigger')'...
Retrieved 5498 rows from database.


In [3]:
all_boundaries_in_database

,fire_id,fire_name,fire_type,ignition_date,capt_date,capt_method,area_ha,perim_km,state,agency,date_retrieved,date_processed,geometry
0,640716,"KOOLEWONG FIRETRAIL, KOOLEWONG",Bushfire,NaT,2026-01-09,None,0.0100000000,0.0300000000,NSW,Rural Fire Service,2026-01-09,2026-01-16,"MULTIPOLYGON (((151.31527 -33.47701, 151.31535..."
1,2b163b9b-2997-4a2e-bb90-30638dbe3ab3,None,None,2025-10-29,2025-11-01,FIREMAPPER,17419.1800000000,126.1000000000,QLD,Qld Fire and Emergency Services,2025-11-02,2025-11-11,"POLYGON ((150.56677 -27.84618, 150.56653 -27.8..."
2,640850,Mayfield,Bushfire,NaT,2026-01-12,None,120.8200000000,16.0400000000,NSW,Rural Fire Service,2026-01-13,2026-01-20,"MULTIPOLYGON (((149.75192 -35.29518, 149.75195..."
3,3806bdcd-c937-48df-a844-033b9661aa98,None,None,2025-10-25,2025-10-25,FIREMAPPER,7066.5300000000,47.4600000000,QLD,Qld Fire and Emergency Services,2025-10-26,2025-11-11,"POLYGON ((145.32980 -16.69327, 145.32967 -16.6..."
4,2b163b9b-2997-4a2e-bb90-30638dbe3ab3,None,None,2025-10-31,2025-10-31,FIREMAPPER,82.5400000000,11.3500000000,QLD,Qld Fire and Emergency Services,2025-11-01,2025-11-11,"POLYGON ((150.65097 -27.72976, 150.65124 -27.7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5493,648841,"Valla Road, Viewmont State Forest Plantations",Prescribed Burn,NaT,2026-03-17,None,99.5700000000,20.0500000000,NSW,Forestry Corporation of NSW,2026-03-17,2026-03-26,"MULTIPOLYGON (((152.88151 -30.61394, 152.88163..."
5494,774925,"SPRINGFIELD, SHIRE OF IRWIN",Bushfire,2026-03-13,2026-03-17,None,857.9100000000,23.6600000000,WA,WA Department of Fire and Emergency Services,2026-03-17,2026-03-26,"MULTIPOLYGON (((114.94776 -29.30700, 114.94761..."
5495,84A9FFED-3594-4587-B0B1-EB2A65B1F219,None,None,2025-12-25,2025-12-25,MICA,0.0100000000,0.0400000000,QLD,Qld Fire and Emergency Services,2025-12-25,2026-03-26,"POLYGON ((150.87639 -28.66811, 150.87629 -28.6..."
5496,649149,Forestry HR Boambee Plantations F26,Prescribed Burn,NaT,2026-03-09,None,52.3300000000,7.1800000000,NSW,Forestry Corporation of NSW,2026-03-10,2026-03-26,"MULTIPOLYGON (((153.04876 -30.30145, 153.04894..."


## pre-filtering

pre-filtering steps on polygons:
- remove any polygons with no area or smaller than 1 ha
- check for duplicates for the same fire and perfrom spatial join on them


In [4]:
#  you can set the minimum area in the config object
mature_fires = database.perform_pre_filter(all_boundaries_in_database)


Started with: 5498
After removing small entries (<1ha): 4584
After removing invalid entries: 4074
After dissolving overlapping entries: 1619
After removing fires processed <60 days ago: 681


/home/jovyan/Burn_severity_polygons/dea-burn-severity/src/dea_burn_severity/database.py:237: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  filtered = filtered.fillna(0)


In [5]:
mature_fires

,geometry,fire_id,fire_name,fire_type,capt_method,area_ha,perim_km,state,agency,date_retrieved,ignition_date,capt_date,date_processed
0,"POLYGON ((144.49985 -37.48609, 144.49970 -37.4...",239457,COFFEYS RD,Current Burnt Area,Satellite Imagery Sentinel,2.9100000000,0.8100000000,VIC,Vic Country Fire Authority,2025-10-20,2025-10-05 00:00:00,2025-10-11 00:00:00,2026-01-21
2,"POLYGON ((152.20279 -28.50077, 152.20406 -28.5...",625194,"Spring Gully, Mount Lindesay Rd, Liston",Bushfire,0,4.0700000000,0.7700000000,NSW,Rural Fire Service,2025-10-20,0,2025-10-19 00:00:00,2026-01-21
3,"POLYGON ((148.85692 -33.21502, 148.85692 -33.2...",641455,"PEABODY RD, MOLONG",Bushfire,0,6.6400000000,1.0400000000,NSW,Rural Fire Service,2026-01-15,0,2026-01-15 00:00:00,2026-01-22
6,"POLYGON ((147.12673 -31.16987, 147.12704 -31.1...",640221,"MURRAWOMBIE RD, GIRILAMBONE",Bushfire,0,16.3800000000,1.5700000000,NSW,Rural Fire Service,2026-01-07,0,2026-01-07 00:00:00,2026-01-14
7,"POLYGON ((151.16715 -23.96762, 151.16706 -23.9...",{7049B9D9-2A6D-47CA-932F-87884F1CA723},0,0,Burnt Area Hub - Bu,57.2400000000,3.6500000000,QLD,Qld Fire and Emergency Services,2025-10-23,2025-10-22 00:00:00,2025-10-22 00:00:00,2026-01-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1607,"POLYGON ((153.02393 -30.48911, 153.02364 -30.4...",625076,"Yellow Rock Island, URUNGA 2455",Bushfire,0,1.1400000000,0.4000000000,NSW,Rural Fire Service,2025-10-20,0,2025-10-20 00:00:00,2026-01-21
1608,"MULTIPOLYGON (((150.69313 -34.45952, 150.69326...",633437,"BONG BONG PASS FIRETRAIL, HUNTLEY",Bushfire,0,1.0800000000,0.4300000000,NSW,Rural Fire Service,2025-11-28,0,2025-11-28 00:00:00,2025-12-05
1610,"POLYGON ((150.69299 -34.45634, 150.69325 -34.4...",633437,"BONG BONG PASS FIRETRAIL, HUNTLEY",Bushfire,0,3.4800000000,0.6800000000,NSW,Rural Fire Service,2025-11-27,0,2025-11-27 00:00:00,2025-12-05
1613,"POLYGON ((152.44336 -28.88612, 152.44346 -28.8...",624959,"KIMS WAY, DRAKE",Bushfire,0,22.3200000000,2.7300000000,NSW,Rural Fire Service,2025-10-20,0,2025-10-19 00:00:00,2026-01-21


# For NOTEBOOK USE ONLY

the next four cells further filter mature fires to process only one day at a time in order to more easily manage workload when conducting manual backfill.
this dosn't need to be copied to the pipline 

In [6]:
processed_dates = list(set(mature_fires['date_processed']))

In [7]:
processed_dates

[datetime.date(2025, 12, 4),
 datetime.date(2026, 1, 15),
 datetime.date(2025, 12, 28),
 datetime.date(2026, 1, 12),
 datetime.date(2025, 12, 5),
 datetime.date(2026, 1, 8),
 datetime.date(2025, 11, 13),
 datetime.date(2026, 1, 5),
 datetime.date(2026, 1, 2),
 datetime.date(2025, 12, 23),
 datetime.date(2025, 12, 10),
 datetime.date(2026, 1, 10),
 datetime.date(2025, 11, 17),
 datetime.date(2025, 12, 13),
 datetime.date(2026, 1, 24),
 datetime.date(2026, 1, 7),
 datetime.date(2025, 12, 11),
 datetime.date(2025, 12, 12),
 datetime.date(2026, 1, 16),
 datetime.date(2025, 12, 20),
 datetime.date(2025, 11, 15),
 datetime.date(2025, 11, 29),
 datetime.date(2025, 12, 24),
 datetime.date(2025, 12, 8),
 datetime.date(2025, 12, 17),
 datetime.date(2025, 11, 19),
 datetime.date(2025, 11, 22),
 datetime.date(2026, 1, 19),
 datetime.date(2025, 11, 23),
 datetime.date(2026, 1, 22),
 datetime.date(2025, 12, 16),
 datetime.date(2025, 11, 27),
 datetime.date(2026, 1, 25),
 datetime.date(2025, 12, 6),


In [8]:
date_to_use = date(2025, 11, 18)
process_today = mature_fires[mature_fires['date_processed'] == date_to_use]

## map burn severity


In [11]:

from dea_burn_severity.burn_severity_processing import BurnSeverityProcessor

burn_processing = BurnSeverityProcessor(config)
burn_processing.process_all_polygons(process_today)

    


All outputs will be saved to: products
Found 4 total features. Processing all of them.

Beginning per-fire processing (single polygon per feature)...

Processing fire polygon: 'fire_id_c73293f5-e78e-46f9-910c-a7b7456472bb'
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Counting good quality pixels for each time step using s2cloudless


/env/lib/python3.10/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


Filtering to 6 out of 8 time steps with at least 99.0% good quality pixels
Applying s2cloudless pixel quality/cloud mask
Returning 6 time steps as a dask array
Attempting load_ard with min_gooddata=0.9 ...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Counting good quality pixels for each time step using s2cloudless
Filtering to 6 out of 9 time steps with at least 90.0% good quality pixels
Applying s2cloudless pixel quality/cloud mask
Returning 6 time steps as a dask array
Success: Loaded 6 time slices.
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir_1', 'nbart_nir_2', 'nbart_swir_2', 'nbart_swir_3', 'oa_nbart_contiguity', 'oa_s2cloudless_mask']
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir_1', 'nbart_nir_2', 'nbart_swir_2', 'nbart_swir_3', 'oa_nbart_contiguity', 'oa_s2cloudless_mask']
Calculating severity based on landcover...
Creating debug/masking layer...
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', '